# Retirement Simulator Quick Start
This notebook demonstrates how to configure and run the Australian Retirement/FIRE simulator.

In [ ]:
import sys
import os
from pathlib import Path

# Move up directories until 'src' or 'pyproject.toml' is found to locate project root
current_path = Path.cwd()
while current_path != current_path.parent:
    if (current_path / 'src').exists() or (current_path / 'pyproject.toml').exists():
        # Change the working directory to the project root
        os.chdir(current_path)
        
        # Add 'src' to sys.path for imports
        src_path = str(current_path / 'src')
        if src_path not in sys.path:
            sys.path.insert(0, src_path)
            
        print(f"Project root found: {current_path}")
        print(f"Working directory set to: {os.getcwd()}")
        break
    current_path = current_path.parent
else:
    raise FileNotFoundError("Could not find project root (containing 'src' or 'pyproject.toml') in parent directories")

In [ ]:
import importlib
import pandas as pd
import numpy as np
import retirement_calculator.simulation as sim_mod
importlib.reload(sim_mod)

from retirement_calculator.models import NREAssetConfig, REAssetConfig, DrawdownPolicy, TrustAssetConfig
from retirement_calculator.rates import ConstantRate

# Use the reloaded module's symbols
simulate = sim_mod.simulate
CalculatorConfig = sim_mod.CalculatorConfig

# Minimal run for focused debug
config = CalculatorConfig(
    current_year=2026,
    current_age=42,
    retirement_age=50,
    super_access_age=60,
    salary=175000.0,
    salary_growth_rate=ConstantRate(0.01),
    initial_super_balance=150000.0,
    inflation_rate=ConstantRate(0.035), # Set to 2.5% to distinguish nominal vs real
    expenses=35000.0,
    retirement_expenses=35000.0,
    drawdown_policy=DrawdownPolicy(mode="rebalanced"),
    nre_config=NREAssetConfig(initial_value=200000.0, annual_contribution=30000.0, capital_gain_component_pct=0.30),
    trust_assets=TrustAssetConfig(
        initial_value=100000.0,
        distribution_yield=0.04,
        annual_capital_gain_dist=0.0,
        growth_rate=ConstantRate(0.05)
    ),
    re_assets=[
        REAssetConfig(
            property_id=1,
            year_bought=2023,
            purchase_price=500000.0,
            valuation_at_base_date=500000.0,
            gross_annual_rent=310 * 52,
            loan_balance=450000.0,
            growth_rate=ConstantRate(0.05),
            sale_year=2036,
            sale_reinvestment_target="nre",
        )
    ],
    surplus_reinvestment_target="cash",
    seed=42,
)

results_df = simulate(config)

# Create requested combined column
results_df['nre_total_income_nominal'] = (
    results_df['nre_ordinary_income_nominal'] + results_df['nre_capital_gain_nominal']
)

# Compute leftover before/after NRE contribution
results_df['leftover_nominal'] = (
    results_df['salary_nominal'] - results_df['personal_income_tax'] - results_df['expenses_nominal'] - results_df.get('re_mortgage_payment_nominal', 0)
)
results_df['leftover_after_nre_nominal'] = (
    results_df['leftover_nominal'] - results_df.get('nre_contribution_nominal', 0)
)

# Select and display the requested columns (first 10 rows)
cols = [
    'year', 'age', 'salary_nominal', 'personal_income_tax', 'expenses_nominal',
    'trust_distribution_income_nominal', 'trust_tax_paid_nominal', 'cash_balance',
    'leftover_nominal', 'leftover_after_nre_nominal'
]

print(results_df[cols].head(10).to_string(index=False))

In [ ]:
df_truncated = results_df.copy()

# Truncate dollar-like numeric columns to 2 decimal places
exclude_cols = {'year', 'age', 'cpi_index'}
cols_to_truncate = [
    col for col in df_truncated.select_dtypes(include='number').columns
    if col not in exclude_cols
]

df_truncated[cols_to_truncate] = (
    np.trunc(df_truncated[cols_to_truncate] * 100) / 100
)

df_truncated[cols].head(20)

## Visualizing Net Worth Projection

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
# Plot net worth (assets minus liabilities) and super, in millions
plt.plot(results_df['age'], results_df['net_worth_real'] / 1e6, label='Net Worth (Real, $M)')
plt.plot(results_df['age'], results_df['super_balance_eoy_real'] / 1e6, label='Super Balance (Real, $M)')
plt.axvline(x=config.retirement_age, color='r', linestyle='--', label='Retirement')
plt.xlabel('Age')
plt.ylabel("Value (Millions of Today's $)")
plt.title('Retirement Projection (Net Worth, Real terms)')
plt.legend()
plt.grid(True)
plt.show()